# 04 · Siting new rapid hubs under budget, capacity, grid and demand uncertainty

**Question.** With a given budget, where should the next 4 × 150 kW hubs go so that the most currently unmet or queue-constrained rapid demand is served within 8 minutes' drive — and which choices stay good if demand is 30 % lower or higher?

**Method.** Candidates = plausible hexes (parking / fuel / retail present or heavy traffic, primary substation not RED), pre-filtered to the 500 with most local unmet demand. Capacitated maximal covering MILP (HiGHS via scipy) with per-hub capacity from Erlang C (P(wait) ≤ 15 % at peak) and a budget; robust two-stage version over P10/P50/P90 scenarios; greedy uncapacitated baseline for contrast.

In [ ]:
import warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from chargescape.utils.paths import PROCESSED, INTERIM, FIGURES, TABLES, DOCS
from chargescape.utils.io import read_gdf
from chargescape.viz.figures import hex_map, save, barh, boroughs
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
feats = read_gdf(PROCESSED / "hex_features_r8.small.parquet")
print(feats.shape)

In [ ]:
meta = json.loads((TABLES / "siting_meta_2026.json").read_text()); print(meta)
sweep = pd.read_csv(TABLES / "siting_budget_sweep_2026.csv"); sweep.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sweep.budget_m, sweep.covered_p50, "o-", label="capacitated MCLP (exact)")
ax.plot(sweep.budget_m, sweep.greedy_uncapacitated_covered, "s--", label="greedy, uncapacitated (optimistic)")
ax.set_xlabel("budget, £M"); ax.set_ylabel("unmet rapid sessions/day served"); ax.set_title("Budget → served unmet demand (2026, P50)", loc="left"); ax.legend(frameon=False); ax.grid(alpha=0.3)
save(fig, "04_budget_curve.png"); plt.show()

## Selected hubs (P50 vs robust) at £20M

In [ ]:
sel = pd.read_parquet(PROCESSED / "siting_selected_2026.small.parquet")
b = 20.0 if (sel.budget_m == 20.0).any() else sel.budget_m.max()
s20 = sel[sel.budget_m == b]
both = set(s20[s20.model == "p50"].h3) & set(s20[s20.model == "robust"].h3)
print(f"budget £{b:.0f}M: {len(s20[s20.model=='p50'])} hubs (P50), {len(s20[s20.model=='robust'])} hubs (robust), {len(both)} in both")
svc = pd.read_parquet(PROCESSED / "hex_service_2026.small.parquet")
g = feats.merge(svc, on="h3", how="left")
g["unmet"] = g["unserved_sessions_day"].fillna(0) + g["rapid_sessions_day"].fillna(0) * g["served_share"].fillna(0) * np.clip(g["expected_wait_min"].fillna(0) / 30, 0, 1)
fig, ax = plt.subplots(figsize=(10, 8))
hex_map(g, "unmet", "Unmet / queue-constrained rapid demand 2026 (sessions/day) with selected hubs", cmap="Greys", scheme="quantiles", k=6, ax=ax, legend_label="sessions/day")
hexes = feats.set_index("h3")
for model, color, marker in [("p50", "#e4572e", "o"), ("robust", "#1b4965", "^")]:
    pts = hexes.loc[s20[s20.model == model].h3]
    gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts.lon, pts.lat), crs=4326).to_crs(27700).plot(ax=ax, color=color, marker=marker, markersize=60, label=f"{model} ({len(pts)})", edgecolor="white")
ax.legend(loc="lower right", frameon=False); save(fig, "04_selected_hubs.png"); plt.show()
s20.groupby(["model", "borough"]).size().unstack(0).fillna(0).astype(int).sort_values("p50", ascending=False).head(15)

## Reading
* The exact capacitated model serves less than the greedy uncapacitated estimate — the gap is the demand that a hub physically cannot absorb at an acceptable wait, which naïve coverage models hand out for free.
* Hubs chosen under demand uncertainty largely overlap with the P50 solution; the differences are the sites whose value depends on the optimistic tail — those are the ones to phase later or de-risk with a smaller configuration.
* Every selected hex carries its primary-substation headroom label, so a RED site never enters the candidate set and loaded primaries pay a connection uplift.